# Feature Vector Extractor
Generates feature vector csv files in the FeatureTrainingData folder

In [ ]:
import subprocess, sys

_packages = [
    'nltk',
    'pandas',
    'scikit-learn',
    'sentence-transformers',
]

for pkg in _packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

In [ ]:
import re
import os
import pandas as pd
from collections import Counter
from nltk.util import ngrams
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

True

## Configuration
Edit the `BOOKS` dictionary to match your `.txt` filenames and genres.
Place all `.txt` files in the same folder as this notebook (or update `BOOKS_DIR`).

In [ ]:
# --- Configuration ---
BOOKS_DIR = "KnownBooks"   # Folder containing the .txt files
EXTRACT_N = 500            # Top n-grams to extract per book (candidates for genre aggregation)
TOP_N     = 100            # Top n-grams to keep per genre (final vocabulary)

BOOKS = {
    # biography (5 books)
    "AutobiographyOfBenjaminFranklin":  "biography",
    "LifeOfKingEdwardVII":              "biography",
    "NarrativeOfFrederickDouglass":     "biography",
    "StoryOfMyLife":                    "biography",
    "AutobiographyOfCharlesDarwin":     "biography",
    # fantasy (5 books)
    "TheWonderfulWizardOfOz":           "fantasy",
    "AliceInWonderland":                "fantasy",
    "ThroughTheLookingGlass":           "fantasy",
    "GrimmsFairyTales":                 "fantasy",
    "GulliversTravels":                 "fantasy",
    # horror (5 books)
    "Frankenstein":                     "horror",
    "CallOfCthulu":                     "horror",
    "Dracula":                          "horror",
    "TalesOfPoe":                       "horror",
    "DrJekyllAndMrHyde":                "horror",
    # romance (5 books)
    "PrideAndPrejudice":                "romance",
    "RomeoAndJuliet":                   "romance",
    "JaneEyre":                         "romance",
    "WutheringHeights":                 "romance",
    "SenseAndSensibility":              "romance",
    # sci-fi (5 books)
    "OnTheTrailOfTheSpacePirates":      "sci-fi",
    "PlagueShip":                       "sci-fi",
    "TheWarOfTheWorlds":                "sci-fi",
    "TheTimeMachine":                   "sci-fi",
    "TwentyThousandLeagues":            "sci-fi",
}

## Clean Raw Text Files (Project Gutenberg)
Strips the standard Project Gutenberg header and footer boilerplate from each `.txt` file
and saves cleaned versions as `<BookName>_clean.txt` in the same directory.
Run this once before extracting n-grams.

In [ ]:
import re
import os

# Project Gutenberg delimiter patterns
START_PATTERN = re.compile(r'\*{3}\s*START OF THE PROJECT GUTENBERG EBOOK[^\n]*\*{3}', re.IGNORECASE)
END_PATTERN   = re.compile(r'\*{3}\s*END OF THE PROJECT GUTENBERG EBOOK[^\n]*\*{3}',   re.IGNORECASE)

def clean_gutenberg(filename):
    """
    Strip Project Gutenberg header and footer from a .txt file.
    Saves the cleaned text as <filename>_clean.txt.
    Returns the cleaned text as a string.
    """
    path = os.path.join(BOOKS_DIR, filename + '.txt')
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        raw = f.read()

    # Find the start marker
    start_match = START_PATTERN.search(raw)
    if start_match:
        text = raw[start_match.end():]
    else:
        print(f'  [WARNING] No START marker found in {filename} — using full text')
        text = raw

    # Find the end marker
    end_match = END_PATTERN.search(text)
    if end_match:
        text = text[:end_match.start()]
    else:
        print(f'  [WARNING] No END marker found in {filename} — keeping text until EOF')

    text = text.strip()

    # Save cleaned version
    out_path = os.path.join(BOOKS_DIR, filename + '_clean.txt')
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(text)

    original_words = len(raw.split())
    cleaned_words  = len(text.split())
    removed_words  = original_words - cleaned_words
    print(f'  {filename}: {original_words:,} → {cleaned_words:,} words  (removed {removed_words:,} boilerplate words)')
    return text

print('Cleaning Project Gutenberg boilerplate...\n')
for book in BOOKS:
    clean_gutenberg(book)

print('\nCleaned files saved as <BookName>_clean.txt')

Cleaning Project Gutenberg boilerplate...

  AutobiographyOfBenjaminFranklin: 79,264 → 76,203 words  (removed 3,061 boilerplate words)
  LifeOfKingEdwardVII: 150,777 → 147,715 words  (removed 3,062 boilerplate words)
  NarrativeOfFrederickDouglass: 43,819 → 40,750 words  (removed 3,069 boilerplate words)
  StoryOfMyLife: 137,927 → 134,873 words  (removed 3,054 boilerplate words)
  AutobiographyOfCharlesDarwin: 25,733 → 22,684 words  (removed 3,049 boilerplate words)
  TheWonderfulWizardOfOz: 42,692 → 39,649 words  (removed 3,043 boilerplate words)
  AliceInWonderland: 29,569 → 26,525 words  (removed 3,044 boilerplate words)
  ThroughTheLookingGlass: 32,789 → 29,752 words  (removed 3,037 boilerplate words)
  GrimmsFairyTales: 104,156 → 101,111 words  (removed 3,045 boilerplate words)
  GulliversTravels: 108,140 → 105,079 words  (removed 3,061 boilerplate words)
  Frankenstein: 78,106 → 75,042 words  (removed 3,064 boilerplate words)
  CallOfCthulu: 15,029 → 11,968 words  (removed 3,061 

## Helper Functions

In [ ]:
nltk.download('words', quiet=True)
from nltk.corpus import words as _nltk_words

# English word whitelist — used to filter non-English tokens in tokenize()
ENGLISH_VOCAB = set(w.lower() for w in _nltk_words.words())
STOP_WORDS    = set(stopwords.words('english'))

def load_text(filename):
    path = os.path.join(BOOKS_DIR, filename + "_clean.txt")
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()

def tokenize(text):
    words = re.findall(r'[a-z]+', text.lower())
    return [w for w in words
            if w not in STOP_WORDS and len(w) > 1 and w in ENGLISH_VOCAB]

def top_ngrams(tokens, n, top_n=TOP_N):
    counts = Counter(ngrams(tokens, n))
    return [(' '.join(gram), count) for gram, count in counts.most_common(top_n)]

print(f'English vocabulary loaded: {len(ENGLISH_VOCAB):,} words')

## Extract N-Grams for All Books

In [ ]:
results = {}  # { book_name: { 'genre': ..., 'unigrams': [...], 'bigrams': [...], 'trigrams': [...] } }

for book, genre in BOOKS.items():
    print(f"Processing: {book} ({genre})...")
    text   = load_text(book)
    tokens = tokenize(text)

    results[book] = {
        'genre':    genre,
        'unigrams': top_ngrams(tokens, 1, top_n=EXTRACT_N),
        'bigrams':  top_ngrams(tokens, 2, top_n=EXTRACT_N),
        'trigrams': top_ngrams(tokens, 3, top_n=EXTRACT_N),
    }

print("\nDone!")

Processing: AutobiographyOfBenjaminFranklin (biography)...
Processing: LifeOfKingEdwardVII (biography)...
Processing: NarrativeOfFrederickDouglass (biography)...
Processing: StoryOfMyLife (biography)...
Processing: AutobiographyOfCharlesDarwin (biography)...
Processing: TheWonderfulWizardOfOz (fantasy)...
Processing: AliceInWonderland (fantasy)...
Processing: ThroughTheLookingGlass (fantasy)...
Processing: GrimmsFairyTales (fantasy)...
Processing: GulliversTravels (fantasy)...
Processing: Frankenstein (horror)...
Processing: CallOfCthulu (horror)...
Processing: Dracula (horror)...
Processing: TalesOfPoe (horror)...
Processing: DrJekyllAndMrHyde (horror)...
Processing: PrideAndPrejudice (romance)...
Processing: RomeoAndJuliet (romance)...
Processing: JaneEyre (romance)...
Processing: WutheringHeights (romance)...
Processing: SenseAndSensibility (romance)...
Processing: OnTheTrailOfTheSpacePirates (sci-fi)...
Processing: PlagueShip (sci-fi)...
Processing: TheWarOfTheWorlds (sci-fi)...
Pr

## Character Name Filtering
Uses NLTK's curated first-name corpus (~7,900 names) to remove any n-gram
whose tokens match a common English given name. This prevents the model from
memorising character names (e.g. *dorothy*, *darcy*, *victor*) instead of
learning genre signals.

In [ ]:
nltk.download('names', quiet=True)
from nltk.corpus import names as nltk_names

PERSON_NAMES = set(n.lower() for n in nltk_names.words())
print(f"Loaded {len(PERSON_NAMES):,} common first names for filtering.")

removal_log = {}

for book in BOOKS:
    removed = {'unigrams': [], 'bigrams': [], 'trigrams': []}
    for ngram_type in ['unigrams', 'bigrams', 'trigrams']:
        kept = []
        for ngram, count in results[book][ngram_type]:
            if any(t in PERSON_NAMES for t in ngram.split()):
                removed[ngram_type].append((ngram, count))
            else:
                kept.append((ngram, count))
        results[book][ngram_type] = kept
    removal_log[book] = removed

total_removed = sum(len(removal_log[b][k]) for b in removal_log for k in removal_log[b])
print(f"Filtering complete — {total_removed} n-grams removed across all books.")

Loaded 7,576 common first names for filtering.
Filtering complete — 6531 n-grams removed across all books.


In [ ]:
os.makedirs('ngrams', exist_ok=True)

rows = []
for book, log in removal_log.items():
    genre = BOOKS[book]
    for ngram_type, items in log.items():
        for ngram, count in items:
            rows.append({'book': book, 'genre': genre,
                         'ngram_type': ngram_type, 'ngram': ngram, 'count': count})

removed_df = pd.DataFrame(rows)
removed_df.to_csv('ngrams/removed_ngrams.csv', index=False)
print(f"Saved {len(removed_df)} removed n-grams to ngrams/removed_ngrams.csv")

Saved 6531 removed n-grams to ngrams/removed_ngrams.csv


## Display Results per Book

In [ ]:
os.makedirs('ngrams', exist_ok=True)

rows = []
for book, data in results.items():
    for ngram_type, key in [('unigrams', 'unigrams'),
                             ('bigrams',  'bigrams'),
                             ('trigrams', 'trigrams')]:
        for rank, (ngram, count) in enumerate(data[key], start=1):
            rows.append({
                'book':       book,
                'genre':      data['genre'],
                'ngram_type': ngram_type,
                'rank':       rank,
                'ngram':      ngram,
                'count':      count,
            })

book_ngrams_df = pd.DataFrame(rows)
book_ngrams_df.to_csv('ngrams/book_ngrams.csv', index=False)
print(f"Saved {len(book_ngrams_df):,} rows to ngrams/book_ngrams.csv")
print(f"  ({len(results)} books × 3 n-gram types × up to {EXTRACT_N} entries each)")

## Genre-Level N-Gram Aggregation
Pools n-gram counts across all books within each genre, then takes the top `TOP_N`
per genre. A word that appears consistently across multiple books in a genre ranks
higher than one that dominates a single book — giving a more representative vocabulary.

In [ ]:
from collections import defaultdict

# Sum n-gram counts across all books within the same genre
genre_counts = defaultdict(lambda: {t: Counter() for t in ['unigrams', 'bigrams', 'trigrams']})

for book, data in results.items():
    genre = data['genre']
    for ngram_type in ['unigrams', 'bigrams', 'trigrams']:
        for ngram, count in data[ngram_type]:
            genre_counts[genre][ngram_type][ngram] += count

# Take top-TOP_N per genre per n-gram type
genre_top = {}
for genre in sorted(genre_counts):
    genre_top[genre] = {
        ngram_type: genre_counts[genre][ngram_type].most_common(TOP_N)
        for ngram_type in ['unigrams', 'bigrams', 'trigrams']
    }

print(f"Genre-level top-{TOP_N} n-grams computed:\n")
for genre in sorted(genre_top):
    u = len(genre_top[genre]['unigrams'])
    b = len(genre_top[genre]['bigrams'])
    t = len(genre_top[genre]['trigrams'])
    print(f"  {genre:12}: {u} unigrams, {b} bigrams, {t} trigrams")

total = sum(len(genre_top[g][t]) for g in genre_top for t in ['unigrams', 'bigrams', 'trigrams'])
print(f"\nTotal vocabulary candidates: {total} (before cross-genre deduplication)")

Genre-level top-100 n-grams computed:

  biography   : 100 unigrams, 100 bigrams, 100 trigrams
  fantasy     : 100 unigrams, 100 bigrams, 100 trigrams
  horror      : 100 unigrams, 100 bigrams, 100 trigrams
  romance     : 100 unigrams, 100 bigrams, 100 trigrams
  sci-fi      : 100 unigrams, 100 bigrams, 100 trigrams

Total vocabulary candidates: 1500 (before cross-genre deduplication)


## N-Gram Overlap & Cross-Genre Filtering
Pairwise heatmaps show how many top-`TOP_N` n-grams each pair of genres shares.
N-grams that appear in **all five genres** carry no discriminative signal and are
removed from `genre_top` before building the vocabulary and feature vectors.

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs('images', exist_ok=True)

genre_list  = sorted(genre_top.keys())
ngram_types = ['unigrams', 'bigrams', 'trigrams']

# ── Pairwise overlap heatmaps ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, ngram_type in zip(axes, ngram_types):
    sets   = {g: set(ng for ng, _ in genre_top[g][ngram_type]) for g in genre_list}
    matrix = [[len(sets[g1] & sets[g2]) for g2 in genre_list] for g1 in genre_list]
    sns.heatmap(matrix, annot=True, fmt='d', cmap='YlOrRd',
                xticklabels=genre_list, yticklabels=genre_list,
                ax=ax, linewidths=0.5, cbar_kws={'label': 'Shared n-grams'})
    ax.set_title(f'{ngram_type.capitalize()} Overlap', fontsize=12)
    ax.tick_params(axis='x', rotation=35)
plt.suptitle(f'Pairwise N-Gram Overlap Between Genres (top-{TOP_N} per genre)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('images/ngram_genre_overlap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: images/ngram_genre_overlap.png')

# ── N-grams common to ALL genres (before removal) ────────────────────────
print('\nN-grams present in ALL 5 genres (non-discriminative):')
for ngram_type in ngram_types:
    sets   = [set(ng for ng, _ in genre_top[g][ngram_type]) for g in genre_list]
    common = set.intersection(*sets)
    sample = sorted(common)[:8]
    suffix = '...' if len(common) > 8 else ''
    print(f'  {ngram_type}: {len(common)} terms — {sample}{suffix}')

# ── Remove n-grams shared across ALL genres ───────────────────────────────
print('\nRemoving cross-genre common n-grams...')
for ngram_type in ngram_types:
    sets       = [set(ng for ng, _ in genre_top[g][ngram_type]) for g in genre_list]
    common_all = set.intersection(*sets)
    for g in genre_list:
        genre_top[g][ngram_type] = [(ng, c) for ng, c in genre_top[g][ngram_type]
                                     if ng not in common_all]
    print(f'  {ngram_type}: removed {len(common_all)} shared terms')

print('\nVocabulary sizes after removal:')
for g in genre_list:
    sizes = {t: len(genre_top[g][t]) for t in ngram_types}
    print(f'  {g:12}: unigrams={sizes["unigrams"]}  bigrams={sizes["bigrams"]}  trigrams={sizes["trigrams"]}')

## Top 10 Unigrams per Genre
Aggregated across all books in each genre — shows the most genre-representative words.

In [ ]:
comparison = {
    genre: [ngram for ngram, _ in genre_top[genre]['unigrams'][:10]]
    for genre in sorted(genre_top)
}

comp_df = pd.DataFrame(comparison)
comp_df.index = [f"#{i+1}" for i in range(10)]
print("Top 10 unigrams per genre (aggregated across books, stop words removed):\n")
display(comp_df)

Top 10 unigrams per genre (aggregated across books, stop words removed):



,biography,fantasy,horror,romance,sci-fi
#1,one,said,one,would,one
#2,great,one,said,could,said
#3,little,could,could,mr,captain
#4,time,would,would,said,could
#5,would,little,upon,one,would
#6,mr,came,us,mrs,time
#7,many,went,must,must,two
#8,made,upon,man,well,us
#9,much,great,time,much,strong
#10,could,time,shall,miss,man


## Export Genre N-Grams to CSV
One CSV per n-gram type, one row per genre-level top-`TOP_N` entry.

In [ ]:
os.makedirs('ngrams', exist_ok=True)

for ngram_type in ['unigrams', 'bigrams', 'trigrams']:
    rows = []
    for genre in sorted(genre_top):
        for rank, (ngram, count) in enumerate(genre_top[genre][ngram_type], start=1):
            rows.append({
                'genre': genre,
                'rank':  rank,
                'ngram': ngram,
                'count': count,
            })
    df = pd.DataFrame(rows)
    out_path = f"ngrams/top100_{ngram_type}.csv"
    df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")

Saved: ngrams/top100_unigrams.csv
Saved: ngrams/top100_bigrams.csv
Saved: ngrams/top100_trigrams.csv


## Export Vocabulary for Classifier
Saves `vocabulary.csv` — the union of genre-level top-`TOP_N` n-grams across all five genres,
de-duplicated. The classifier notebook uses this list to build its TF-IDF feature matrix.

In [ ]:
# Union of genre-level top-100 n-grams across all genres
all_ngrams = set()
for data in genre_top.values():
    for word,   _ in data['unigrams']:  all_ngrams.add(word)
    for phrase, _ in data['bigrams']:   all_ngrams.add(phrase)
    for phrase, _ in data['trigrams']:  all_ngrams.add(phrase)

n_uni = len({w for data in genre_top.values() for w, _ in data['unigrams']})
n_bi  = len({p for data in genre_top.values() for p, _ in data['bigrams']})
n_tri = len({p for data in genre_top.values() for p, _ in data['trigrams']})

vocab_df = pd.DataFrame(sorted(all_ngrams), columns=['word'])
vocab_df.to_csv('FeatureTrainingData/vocabulary.csv', index=False)
print(f"Vocabulary size: {len(vocab_df)} entries  ({n_uni} unigrams + {n_bi} bigrams + {n_tri} trigrams)")
print(f"Saved to 'FeatureTrainingData/vocabulary.csv'")

Vocabulary size: 1109 entries  (214 unigrams + 399 bigrams + 496 trigrams)
Saved to 'FeatureTrainingData/vocabulary.csv'


## Build Feature Vector CSV
Applies TF-IDF (restricted to the vocabulary) to every labelled text partition
and saves the result as a ready-to-train feature matrix.

**Requires:** `TestingData/partitions.csv` from `Book partitioner.ipynb`

**Outputs:**
- `FeatureTrainingData/featurevector.csv` — one row per partition, columns for `partition_id`, `genre`, and one TF-IDF score per vocabulary entry (unigrams + bigrams + trigrams)
- `FeatureTrainingData/vectorizer.pkl` — fitted vectorizer for transforming new unseen text

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

# Load labelled partitions (produced by Book partitioner.ipynb)
partitions = pd.read_csv('TestingData/partitions.csv')

# Full vocabulary: unigrams + bigrams + trigrams in sorted order (same as vocabulary.csv)
vocab_list = sorted(all_ngrams)

# Fit TF-IDF restricted to the shared vocabulary.
# ngram_range=(1,3) enables phrase matching; stop_words removes stopwords before
# generating n-grams so phrases match the stopword-filtered extraction logic.
vectorizer = TfidfVectorizer(
    vocabulary    = vocab_list,
    ngram_range   = (1, 3),
    stop_words    = 'english',
    sublinear_tf  = True,
    strip_accents = 'unicode',
    analyzer      = 'word',
    token_pattern = r'[a-z]+',
    lowercase     = True,
)

X = vectorizer.fit_transform(partitions['text'])
feature_names = vectorizer.get_feature_names_out()

# Assemble: partition_id | genre | feature_1 | feature_2 | ...
feat_df = pd.DataFrame(X.toarray(), columns=feature_names)
feat_df.insert(0, 'genre',        partitions['genre'].values)
feat_df.insert(0, 'partition_id', partitions['partition_id'].values)

feat_df.to_csv('FeatureTrainingData/featurevector.csv', index=False)
joblib.dump(vectorizer, 'FeatureTrainingData/vectorizer.pkl')

print(f"Saved: FeatureTrainingData/featurevector.csv")
print(f"  {feat_df.shape[0]} partitions  ×  {len(feature_names)} features")
print(f"  ({n_uni} unigram + {n_bi} bigram + {n_tri} trigram features)")
print(f"Saved: FeatureTrainingData/vectorizer.pkl")
feat_df.iloc[:3, :8]   # preview first few columns


Saved: FeatureTrainingData/featurevector.csv
  5000 partitions  ×  1109 features
  (214 unigram + 399 bigram + 496 trigram features)
Saved: FeatureTrainingData/vectorizer.pkl


,partition_id,genre,accompanied princess,accompanied princess wales,across,actions man man,ad le,ad le came
0,AutobiographyOfBenjaminFranklin_partition_a,biography,0.0,0.0,0.0,0.0,0.0,0.0
1,AutobiographyOfBenjaminFranklin_partition_b,biography,0.0,0.0,0.0,0.0,0.0,0.0
2,AutobiographyOfBenjaminFranklin_partition_c,biography,0.0,0.0,0.0,0.0,0.0,0.0


## Save TF-IDF with Book Column
Adds a `book` column to the existing TF-IDF feature matrix for consistency with other feature files.

In [ ]:
# Save tfidf_features.csv: adds 'book' column to featurevector.csv for consistency
tfidf_out = feat_df.copy()
tfidf_out.insert(1, 'book', partitions['book'].values)
tfidf_out.to_csv('FeatureTrainingData/tfidf_features.csv', index=False)
print(f"Saved: FeatureTrainingData/tfidf_features.csv  {tfidf_out.shape}")

Saved: FeatureTrainingData/tfidf_features.csv  (5000, 1112)


## Bag-of-Words (Count Vectors)
Builds raw term-count vectors using the same genre vocabulary used for TF-IDF.
Outputs `FeatureTrainingData/bow_features.csv`.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

bow_vec = CountVectorizer(
    vocabulary    = vocab_list,
    ngram_range   = (1, 3),
    stop_words    = 'english',
    strip_accents = 'unicode',
    analyzer      = 'word',
    token_pattern = r'[a-z]+',
    lowercase     = True,
)
X_bow = bow_vec.fit_transform(partitions['text'])
bow_feature_names = bow_vec.get_feature_names_out()

bow_df = pd.DataFrame(X_bow.toarray(), columns=bow_feature_names)
bow_df.insert(0, 'genre',        partitions['genre'].values)
bow_df.insert(0, 'book',         partitions['book'].values)
bow_df.insert(0, 'partition_id', partitions['partition_id'].values)
bow_df.to_csv('FeatureTrainingData/bow_features.csv', index=False)
print(f"Saved: FeatureTrainingData/bow_features.csv")
print(f"  Shape: {bow_df.shape[0]} partitions x {len(bow_feature_names)} features")
bow_df.iloc[:3, :8]

Saved: FeatureTrainingData/bow_features.csv
  Shape: 5000 partitions x 1109 features


,partition_id,book,genre,accompanied princess,accompanied princess wales,across,actions man man,ad le
0,AutobiographyOfBenjaminFranklin_partition_a,AutobiographyOfBenjaminFranklin,biography,0,0,0,0,0
1,AutobiographyOfBenjaminFranklin_partition_b,AutobiographyOfBenjaminFranklin,biography,0,0,0,0,0
2,AutobiographyOfBenjaminFranklin_partition_c,AutobiographyOfBenjaminFranklin,biography,0,0,0,0,0


## Stylometric Features
Captures surface writing-style signals: sentence length, word length distribution, punctuation density, type-token ratio, hapax ratio, and syllable complexity.
Outputs `FeatureTrainingData/stylometric_features.csv`.

In [ ]:
import re as _re
import numpy as _np
from collections import Counter as _Ctr

def _syllables(word):
    w, vowels, count, prev = word.lower(), 'aeiouy', 0, False
    for ch in w:
        v = ch in vowels
        if v and not prev:
            count += 1
        prev = v
    return max(1, count)

def stylometric_features(text):
    words = _re.findall(r'[a-zA-Z]+', text)
    sents = [s.strip() for s in _re.split(r'[.!?]+', text) if s.strip()]
    if not words:
        return dict.fromkeys(['avg_word_len','avg_sent_len','sent_len_std',
                               'type_token_ratio','punct_density','comma_ratio',
                               'semicolon_ratio','quote_density','avg_syllables',
                               'hapax_ratio'], 0.0)
    wl   = [len(w) for w in words]
    sl   = [len(_re.findall(r'[a-zA-Z]+', s)) for s in sents]
    sl   = [x for x in sl if x > 0] or [len(words)]
    uniq = set(w.lower() for w in words)
    freq = _Ctr(w.lower() for w in words)
    hapax = sum(1 for f in freq.values() if f == 1)
    return {
        'avg_word_len':     float(_np.mean(wl)),
        'avg_sent_len':     float(_np.mean(sl)),
        'sent_len_std':     float(_np.std(sl)),
        'type_token_ratio': len(uniq) / len(words),
        'punct_density':    len(_re.findall(r'[.,!?;:\'"()\-]', text)) / max(len(words), 1),
        'comma_ratio':      text.count(',') / max(len(words), 1),
        'semicolon_ratio':  text.count(';') / max(len(words), 1),
        'quote_density':    (text.count('"') + text.count("'")) / max(len(words), 1),
        'avg_syllables':    float(_np.mean([_syllables(w) for w in words])),
        'hapax_ratio':      hapax / max(len(uniq), 1),
    }

print("Computing stylometric features for", len(partitions), "partitions...")
stylo_rows = []
for i, row in partitions.iterrows():
    feats = stylometric_features(row['text'])
    feats.update({'partition_id': row['partition_id'],
                  'book': row['book'], 'genre': row['genre']})
    stylo_rows.append(feats)
    if (i + 1) % 1000 == 0:
        print(f"  {i+1}/{len(partitions)}")

stylo_df  = pd.DataFrame(stylo_rows)
meta_cols = ['partition_id', 'book', 'genre']
feat_cols = [c for c in stylo_df.columns if c not in meta_cols]
stylo_df  = stylo_df[meta_cols + feat_cols]
stylo_df.to_csv('FeatureTrainingData/stylometric_features.csv', index=False)
print(f"Saved: FeatureTrainingData/stylometric_features.csv  {stylo_df.shape}")
stylo_df.head(3)

Computing stylometric features for 5000 partitions...
  1000/5000
  2000/5000
  3000/5000
  4000/5000
  5000/5000
Saved: FeatureTrainingData/stylometric_features.csv  (5000, 13)


,partition_id,book,genre,avg_word_len,avg_sent_len,sent_len_std,type_token_ratio,punct_density,comma_ratio,semicolon_ratio,quote_density,avg_syllables,hapax_ratio
0,AutobiographyOfBenjaminFranklin_partition_a,AutobiographyOfBenjaminFranklin,biography,3.913043,23.000000,17.713774,0.495169,0.190821,0.089372,0.014493,0.031401,1.427536,0.731707
1,AutobiographyOfBenjaminFranklin_partition_b,AutobiographyOfBenjaminFranklin,biography,4.721393,14.357143,11.013674,0.562189,0.144279,0.054726,0.000000,0.012438,1.706468,0.783186
2,AutobiographyOfBenjaminFranklin_partition_c,AutobiographyOfBenjaminFranklin,biography,4.473945,25.187500,25.372768,0.550868,0.151365,0.069479,0.004963,0.024814,1.575682,0.747748


## Lexical-Morphological Features (LMF)
Uses NLTK POS tagging to compute POS-category ratios (nouns, verbs, adjectives, adverbs), content-word ratio, function-word ratio, and lexical density.
Outputs `FeatureTrainingData/lmf_features.csv`.

In [ ]:
import numpy as np

nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

_STOP  = set(stopwords.words('english'))
_NOUN  = {'NN','NNS','NNP','NNPS'}
_VERB  = {'VB','VBD','VBG','VBN','VBP','VBZ'}
_ADJ   = {'JJ','JJR','JJS'}
_ADV   = {'RB','RBR','RBS'}
_CONT  = _NOUN | _VERB | _ADJ | _ADV

def lmf_features(text):
    alpha = re.findall(r'[a-zA-Z]+', text)
    if not alpha:
        return dict.fromkeys(['noun_ratio','verb_ratio','adj_ratio','adv_ratio',
                              'content_ratio','function_ratio','lexical_density',
                              'avg_morph_len','morph_len_var'], 0.0)
    pos = nltk.pos_tag(alpha)
    n   = len(alpha)
    wl  = [len(w) for w in alpha]
    mean_wl = sum(wl) / n
    return {
        'noun_ratio':      sum(1 for _,t in pos if t in _NOUN) / n,
        'verb_ratio':      sum(1 for _,t in pos if t in _VERB) / n,
        'adj_ratio':       sum(1 for _,t in pos if t in _ADJ)  / n,
        'adv_ratio':       sum(1 for _,t in pos if t in _ADV)  / n,
        'content_ratio':   sum(1 for _,t in pos if t in _CONT) / n,
        'function_ratio':  sum(1 for w,_ in pos if w.lower() in _STOP) / n,
        'lexical_density': sum(1 for _,t in pos if t in _CONT) / n,
        'avg_morph_len':   mean_wl,
        'morph_len_var':   sum((x - mean_wl)**2 for x in wl) / n,
    }

# Sanity check before the full loop
_test = lmf_features("The quick brown fox jumps over the lazy dog")
print("Sanity check:", {k: round(v, 3) for k, v in _test.items()})

print(f"\nComputing LMF features ({len(partitions)} partitions — may take 5-10 min)...")
lmf_rows = []
_errors  = 0
for i, row in partitions.iterrows():
    try:
        feats = lmf_features(row['text'])
    except Exception as e:
        feats = dict.fromkeys(['noun_ratio','verb_ratio','adj_ratio','adv_ratio',
                               'content_ratio','function_ratio','lexical_density',
                               'avg_morph_len','morph_len_var'], 0.0)
        _errors += 1
        if _errors <= 3:
            print(f"  [WARN] row {i}: {e}")
    feats.update({'partition_id': row['partition_id'],
                  'book': row['book'], 'genre': row['genre']})
    lmf_rows.append(feats)
    if (i + 1) % 500 == 0:
        print(f"  {i+1}/{len(partitions)}")

if _errors:
    print(f"  [WARN] {_errors} rows fell back to zeros")

lmf_df    = pd.DataFrame(lmf_rows)
meta_cols = ['partition_id', 'book', 'genre']
feat_cols = [c for c in lmf_df.columns if c not in meta_cols]
lmf_df    = lmf_df[meta_cols + feat_cols]
lmf_df.to_csv('FeatureTrainingData/lmf_features.csv', index=False)
print(f"Saved: FeatureTrainingData/lmf_features.csv  {lmf_df.shape}")
lmf_df.head(3)

Sanity check: {'noun_ratio': 0.333, 'verb_ratio': 0.111, 'adj_ratio': 0.222, 'adv_ratio': 0.0, 'content_ratio': 0.667, 'function_ratio': 0.333, 'lexical_density': 0.667, 'avg_morph_len': 3.889, 'morph_len_var': 0.765}

Computing LMF features (5000 partitions — may take 5-10 min)...
  500/5000
  1000/5000
  1500/5000
  2000/5000
  2500/5000
  3000/5000
  3500/5000
  4000/5000
  4500/5000
  5000/5000
Saved: FeatureTrainingData/lmf_features.csv  (5000, 12)


,partition_id,book,genre,noun_ratio,verb_ratio,adj_ratio,adv_ratio,content_ratio,function_ratio,lexical_density,avg_morph_len,morph_len_var
0,AutobiographyOfBenjaminFranklin_partition_a,AutobiographyOfBenjaminFranklin,biography,0.214976,0.185990,0.050725,0.055556,0.507246,0.574879,0.507246,3.913043,5.369250
1,AutobiographyOfBenjaminFranklin_partition_b,AutobiographyOfBenjaminFranklin,biography,0.308458,0.144279,0.089552,0.039801,0.582090,0.490050,0.582090,4.721393,8.623871
2,AutobiographyOfBenjaminFranklin_partition_c,AutobiographyOfBenjaminFranklin,biography,0.235732,0.168734,0.076923,0.042184,0.523573,0.543424,0.523573,4.473945,6.353540


## SBERT Sentence Embeddings
Encodes each 400-word partition with `all-MiniLM-L6-v2` producing 384-dimensional dense semantic vectors.
Outputs `FeatureTrainingData/sbert_features.npy` (binary) and `.csv` (with metadata).

In [ ]:
import numpy as np

try:
    from sentence_transformers import SentenceTransformer

    print("Loading SBERT model (all-MiniLM-L6-v2)...")
    sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

    texts = partitions['text'].tolist()
    print(f"Encoding {len(texts):,} partitions (batch_size=64)...")
    sbert_emb = sbert_model.encode(texts, batch_size=64,
                                    show_progress_bar=True, convert_to_numpy=True)
    print(f"Embeddings shape: {sbert_emb.shape}")

    np.save('FeatureTrainingData/sbert_features.npy', sbert_emb)

    sbert_cols = [f'sbert_{i}' for i in range(sbert_emb.shape[1])]
    sbert_df   = pd.DataFrame(sbert_emb, columns=sbert_cols)
    sbert_df.insert(0, 'genre',        partitions['genre'].values)
    sbert_df.insert(0, 'book',         partitions['book'].values)
    sbert_df.insert(0, 'partition_id', partitions['partition_id'].values)
    sbert_df.to_csv('FeatureTrainingData/sbert_features.csv', index=False)
    print(f"Saved: FeatureTrainingData/sbert_features.npy  ({sbert_emb.nbytes/1e6:.1f} MB)")
    print(f"Saved: FeatureTrainingData/sbert_features.csv")

except ImportError:
    print("sentence_transformers not installed — SBERT skipped.")
    print("To enable: pip install sentence-transformers")
    print("SBERT feature files will not be created; Clustering.ipynb will fall back to TF-IDF-PCA50.")

Loading SBERT model (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding 5,000 partitions (batch_size=64)...


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Embeddings shape: (5000, 384)
Saved: FeatureTrainingData/sbert_features.npy  (7.7 MB)
Saved: FeatureTrainingData/sbert_features.csv


## Feature Files Summary

All feature representations saved to `FeatureTrainingData/`:

| File | Description | Features |
|------|-------------|----------|
| `bow_features.csv` | Bag-of-Words count vectors | 1109 n-gram counts |
| `tfidf_features.csv` | TF-IDF weighted n-gram vectors | 1109 TF-IDF scores |
| `stylometric_features.csv` | Surface writing-style signals | 10 numeric |
| `lmf_features.csv` | POS-based lexical-morphological ratios | 9 numeric |
| `sbert_features.npy` | SBERT embeddings (fast binary load) | 384 float32 |
| `sbert_features.csv` | Same embeddings with partition metadata | 384 dims |

**Run order:** `Book partitioner.ipynb` → **`BOW.ipynb`** → `Clustering.ipynb`

## AllBooks Corpus — Feature Vectors
Uses the curated vocabulary and fitted vectorizers from the KnownBooks section above
to extract consistent feature representations for all ~97 books in `AllBooks/<genre>/`.

Saved to `FeatureTrainingData/`:
- `allbooks_partitions.csv` — partition metadata + raw text (200 × 400-word samples per book)
- `allbooks_bow_features.csv` — curated-vocabulary count vectors
- `allbooks_tfidf_features.csv` — curated-vocabulary TF-IDF vectors
- `allbooks_stylometric_features.csv` — surface writing-style signals (10 features)
- `allbooks_lmf_features.csv` — POS-based lexical-morphological ratios (9 features) *(slow: ~30 min)*
- `allbooks_sbert_features.npy/.csv` — SBERT 384-d semantic embeddings

`ClusterAllBooks.ipynb` loads these files instead of re-extracting features at runtime.

In [ ]:
import random
from pathlib import Path as _Path

ALLBOOKS_DIR           = 'AllBooks'
AB_PARTITIONS_PER_BOOK = 200
AB_WORDS_PER_PARTITION = 400
AB_SEED                = 42
AB_GENRES              = ['biography', 'fantasy', 'horror', 'romance', 'sci-fi']

_AB_START = re.compile(r'\*{3}\s*START OF THE PROJECT GUTENBERG EBOOK[^\n]*\*{3}', re.IGNORECASE)
_AB_END   = re.compile(r'\*{3}\s*END OF THE PROJECT GUTENBERG EBOOK[^\n]*\*{3}',   re.IGNORECASE)

def _strip_pg(raw):
    m = _AB_START.search(raw)
    t = raw[m.end():] if m else raw
    m = _AB_END.search(t)
    return t[:m.start()].strip() if m else t.strip()

def _is_english(text, threshold=0.60, sample_tokens=300):
    """Return True if ≥ threshold fraction of sampled tokens are in ENGLISH_VOCAB."""
    tokens = re.findall(r'[a-z]+', text.lower()[:10000])
    if len(tokens) < 50:
        return True
    sample = tokens[:sample_tokens]
    ratio  = sum(1 for w in sample if w in ENGLISH_VOCAB) / len(sample)
    return ratio >= threshold

# ── 1. Partition AllBooks (deterministic; cached after first run) ──────────
AB_PARTITIONS_CSV = 'FeatureTrainingData/allbooks_partitions.csv'
if os.path.exists(AB_PARTITIONS_CSV):
    ab_df = pd.read_csv(AB_PARTITIONS_CSV)
    print(f'Loaded {len(ab_df):,} cached partitions from {AB_PARTITIONS_CSV}')
else:
    _rng, records, skipped = random.Random(AB_SEED), [], []
    for genre in AB_GENRES:
        gdir = _Path(ALLBOOKS_DIR) / genre
        if not gdir.exists():
            print(f'  [WARN] {gdir} not found'); continue
        for fp in sorted(gdir.glob('*.txt')):
            try:   raw = fp.read_text(encoding='utf-8',  errors='ignore')
            except: raw = fp.read_text(encoding='latin-1', errors='ignore')
            text  = _strip_pg(raw)
            if not _is_english(text):
                print(f'  [SKIP] {fp.name} — non-English')
                skipped.append(fp.name)
                continue
            toks  = text.split()
            if len(toks) < AB_WORDS_PER_PARTITION + 50:
                print(f'  [SKIP] {fp.name} — {len(toks)} words'); continue
            max_s = len(toks) - AB_WORDS_PER_PARTITION
            n     = min(AB_PARTITIONS_PER_BOOK, max_s)
            for i, s in enumerate(_rng.sample(range(max_s + 1), n)):
                records.append({'partition_id': f'{fp.stem}_part_{i:03d}',
                                'book': fp.stem, 'genre': genre,
                                'text': ' '.join(toks[s:s + AB_WORDS_PER_PARTITION])})
    ab_df = pd.DataFrame(records)
    ab_df.to_csv(AB_PARTITIONS_CSV, index=False)
    print(f'Partitioned {len(ab_df):,} AllBooks texts → {AB_PARTITIONS_CSV}')
    if skipped:
        print(f'Skipped {len(skipped)} non-English books: {skipped}')

print(ab_df['genre'].value_counts().to_string())
ab_texts = ab_df['text'].tolist()

# ── 2. BOW features (curated vocabulary) ─────────────────────────────────
X_ab_bow  = bow_vec.transform(ab_texts)
ab_bow_df = pd.DataFrame(X_ab_bow.toarray(), columns=bow_vec.get_feature_names_out())
ab_bow_df.insert(0, 'genre', ab_df['genre'].values)
ab_bow_df.insert(0, 'book',  ab_df['book'].values)
ab_bow_df.insert(0, 'partition_id', ab_df['partition_id'].values)
ab_bow_df.to_csv('FeatureTrainingData/allbooks_bow_features.csv', index=False)
print(f'Saved: allbooks_bow_features.csv  {ab_bow_df.shape}')

# ── 3. TF-IDF features (curated vocabulary + sublinear_tf) ───────────────
X_ab_tfidf  = vectorizer.transform(ab_texts)
ab_tfidf_df = pd.DataFrame(X_ab_tfidf.toarray(), columns=vectorizer.get_feature_names_out())
ab_tfidf_df.insert(0, 'genre', ab_df['genre'].values)
ab_tfidf_df.insert(0, 'book',  ab_df['book'].values)
ab_tfidf_df.insert(0, 'partition_id', ab_df['partition_id'].values)
ab_tfidf_df.to_csv('FeatureTrainingData/allbooks_tfidf_features.csv', index=False)
print(f'Saved: allbooks_tfidf_features.csv  {ab_tfidf_df.shape}')

# ── 4. Stylometric features ───────────────────────────────────────────────
print(f'Computing stylometric features ({len(ab_texts):,} partitions)...')
ab_stylo_rows = [stylometric_features(t) for t in ab_texts]
ab_stylo_df   = pd.DataFrame(ab_stylo_rows)
ab_stylo_df.insert(0, 'genre', ab_df['genre'].values)
ab_stylo_df.insert(0, 'book',  ab_df['book'].values)
ab_stylo_df.insert(0, 'partition_id', ab_df['partition_id'].values)
ab_stylo_df.to_csv('FeatureTrainingData/allbooks_stylometric_features.csv', index=False)
print(f'Saved: allbooks_stylometric_features.csv  {ab_stylo_df.shape}')

In [ ]:
# ── LMF features for AllBooks (slow: ~20-40 min for ~19k partitions) ─────
print(f'Computing LMF features for {len(ab_texts):,} partitions...')
print('This may take 20-40 minutes.\n')
ab_lmf_rows = []
_lmf_errors = 0
for _i, _t in enumerate(ab_texts):
    try:
        _feats = lmf_features(_t)
    except Exception:
        _feats = dict.fromkeys(['noun_ratio','verb_ratio','adj_ratio','adv_ratio',
                                'content_ratio','function_ratio','lexical_density',
                                'avg_morph_len','morph_len_var'], 0.0)
        _lmf_errors += 1
    _feats['partition_id'] = ab_df.iloc[_i]['partition_id']
    _feats['book']         = ab_df.iloc[_i]['book']
    _feats['genre']        = ab_df.iloc[_i]['genre']
    ab_lmf_rows.append(_feats)
    if (_i + 1) % 2000 == 0:
        print(f'  {_i+1}/{len(ab_texts)}')

if _lmf_errors:
    print(f'  [WARN] {_lmf_errors} rows fell back to zeros')

ab_lmf_df = pd.DataFrame(ab_lmf_rows)
_lmf_meta = ['partition_id', 'book', 'genre']
ab_lmf_df = ab_lmf_df[_lmf_meta + [c for c in ab_lmf_df.columns if c not in _lmf_meta]]
ab_lmf_df.to_csv('FeatureTrainingData/allbooks_lmf_features.csv', index=False)
print(f'Saved: allbooks_lmf_features.csv  {ab_lmf_df.shape}')

In [ ]:
import numpy as np

# ── SBERT embeddings for AllBooks ────────────────────────────────────────
try:
    from sentence_transformers import SentenceTransformer
    print('Loading SBERT (all-MiniLM-L6-v2)...')
    _ab_sbert = SentenceTransformer('all-MiniLM-L6-v2')
    print(f'Encoding {len(ab_texts):,} partitions (batch_size=128)...')
    ab_sbert_emb = _ab_sbert.encode(ab_texts, batch_size=128,
                                     show_progress_bar=True, convert_to_numpy=True)
    np.save('FeatureTrainingData/allbooks_sbert_features.npy', ab_sbert_emb)
    _sbert_cols = [f'sbert_{i}' for i in range(ab_sbert_emb.shape[1])]
    ab_sbert_df = pd.DataFrame(ab_sbert_emb, columns=_sbert_cols)
    ab_sbert_df.insert(0, 'genre',        ab_df['genre'].values)
    ab_sbert_df.insert(0, 'book',         ab_df['book'].values)
    ab_sbert_df.insert(0, 'partition_id', ab_df['partition_id'].values)
    ab_sbert_df.to_csv('FeatureTrainingData/allbooks_sbert_features.csv', index=False)
    print(f'Saved: allbooks_sbert_features.npy  ({ab_sbert_emb.nbytes/1e6:.1f} MB)')
    print(f'Saved: allbooks_sbert_features.csv  {ab_sbert_df.shape}')
except ImportError:
    print('sentence-transformers not installed — SBERT skipped.')
    print('Run: pip install sentence-transformers')